# Week 10 · Day 1 (TensorFlow · VGG16) — Transfer Learning: Feature Extraction

Same idea, in **TensorFlow / Keras**, using the **VGG16** backbone.

- Reuse a CNN pretrained on **ImageNet** instead of training from scratch.
- **VGG16** = a deep stack of 3×3 convolutions (AlexNet's famous successor); simple and a great feature extractor.
- **Feature extraction:** freeze the pretrained backbone, train only a new head.
- Dataset: **FER2013** (7 emotions).

> **Kaggle GPU:** Settings → Accelerator → GPU, then add the FER2013 dataset via Add Input.  
> Keras uses the GPU automatically — no manual device moves.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

tf.random.set_seed(42)
print("TF version:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices("GPU"))

## 1. Load FER2013 with `image_dataset_from_directory`

- Keras reads class-folder datasets directly — each subfolder name is a class.
- Load at **224×224** (VGG16's expected input size), as **RGB** (3 channels).

In [ ]:
# Kaggle paths — folders containing the class subfolders
TRAIN_DIR = "/kaggle/input/fer2013/train"
TEST_DIR  = "/kaggle/input/fer2013/test"

IMG_SIZE = (224, 224)
BATCH = 64

train_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH,
    label_mode="int", color_mode="rgb", shuffle=True)
test_ds = keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH,
    label_mode="int", color_mode="rgb", shuffle=False)

class_names = train_ds.class_names
n_classes = len(class_names)
print("classes:", class_names)

In [ ]:
# prefetch keeps the GPU fed while the CPU loads the next batch
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

## 2. Load the pretrained VGG16 (backbone only)

- `VGG16(weights="imagenet", include_top=False)` loads the ImageNet backbone **without** its 1000-class head.
- `include_top=False` = "feature extractor only" — we add our own head next.

In [ ]:
base_model = keras.applications.VGG16(
    weights="imagenet",
    include_top=False,               # drop the ImageNet head
    input_shape=(224, 224, 3))

print("VGG16 backbone loaded. output feature-map shape:", base_model.output_shape)

## 3. Freeze the backbone, add a new head

- `base_model.trainable = False` → freeze all VGG16 weights.
- Add: **preprocessing → backbone → pooling → new Dense head** for 7 classes.
- `vgg16.preprocess_input` scales pixels the way ImageNet VGG16 expects.

In [ ]:
base_model.trainable = False    # freeze the backbone

inputs = keras.Input(shape=(224, 224, 3))
x = keras.applications.vgg16.preprocess_input(inputs)      # ImageNet preprocessing for VGG16
x = base_model(x, training=False)                          # frozen backbone
x = keras.layers.GlobalAveragePooling2D()(x)               # feature map -> vector
outputs = keras.layers.Dense(n_classes, activation="softmax")(x)   # new head

model = keras.Model(inputs, outputs)
model.summary()

- In the summary, **most parameters are "Non-trainable"** (the frozen VGG16 backbone).
- Only the small Dense head trains — that's feature extraction.

## 4. Train only the head

- `compile` picks the loss + optimizer.
- `.fit()` runs the whole training loop — the four moves happen inside.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",   # integer labels
    metrics=["accuracy"])

history = model.fit(train_ds, validation_data=test_ds, epochs=5)

## 5. Evaluate

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

tl_loss, tl_acc = model.evaluate(test_ds, verbose=0)
print(f"transfer-learning test accuracy: {tl_acc:.2%}")

preds, trues = [], []
for xb, yb in test_ds:
    preds.extend(model.predict(xb, verbose=0).argmax(1))
    trues.extend(yb.numpy())

cm = confusion_matrix(trues, preds)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
plt.title(f"VGG16 feature extraction on FER2013 — {tl_acc:.1%}")
plt.tight_layout()
plt.show()

## 6. The rematch: transfer learning vs from-scratch

- Week 9's from-scratch CNN on FER2013: **~45%** (use your class's actual number).
- Frozen pretrained VGG16 + a small head should beat it, with less training.
- **Why:** VGG16's ImageNet features already capture the edges and shapes faces are made of.

In [ ]:
scratch_acc = 0.45   # <- your Week 9 from-scratch result

plt.bar(["From scratch\n(Week 9)", "VGG16 transfer\n(today)"],
        [scratch_acc * 100, tl_acc * 100], color=["gray", "green"])
plt.ylabel("test accuracy (%)")
plt.title("Same data, same effort — reuse wins")
for i, v in enumerate([scratch_acc * 100, tl_acc * 100]):
    plt.text(i, v + 1, f"{v:.0f}%", ha="center")
plt.show()

## Your turn (solo task) ✍️

- Swap VGG16 for **MobileNetV2** and compare accuracy **and** speed (MobileNet is much smaller).
- Starter below — same steps: load backbone → freeze → add head.
- Report: which is more accurate? which trains faster?

In [ ]:
# ===== YOUR CODE (solo task) =====
# hint:
# base = keras.applications.MobileNetV2(weights="imagenet", include_top=False, input_shape=(224,224,3))
# base.trainable = False
# inputs = keras.Input((224,224,3))
# x = keras.applications.mobilenet_v2.preprocess_input(inputs)
# x = base(x, training=False)
# x = keras.layers.GlobalAveragePooling2D()(x)
# outputs = keras.layers.Dense(n_classes, activation="softmax")(x)
# m = keras.Model(inputs, outputs)
# ... compile + fit as above


## Summary

- **Feature extraction with VGG16:** load with `include_top=False`, set `trainable = False`, add pooling + a new `Dense` head.
- `vgg16.preprocess_input` handles ImageNet-style scaling; `image_dataset_from_directory` loads class folders directly.
- `.fit()` trains only the head (the backbone is frozen).
- Beats the from-scratch CNN on FER2013 — the core transfer-learning lesson.
- Keras uses the GPU automatically.

> **Note:** VGG16 is a large, slower backbone (lots of parameters). MobileNet (solo task) is far lighter — a good comparison of the accuracy-vs-speed trade-off.

**Tomorrow:** fine-tuning (unfreeze the backbone) + a first look at object detection.